# R²D-HOPE-MoRE — Benchmark vs SOTA & Commercial Models

Evaluates on 4 axes:
1. **Perplexity** on WikiText-103 (language modelling quality)
2. **Throughput** (tokens/sec at batch=1 and batch=8)
3. **Parameter efficiency** (PPL per million params)
4. **Memory footprint** (peak VRAM at inference)

**Comparison targets** (all run locally on Colab GPU, no API keys needed):

| Model | Params | Source |
|---|---|---|
| R²D-HOPE-MoRE (ours, small) | ~6.5M | local checkpoint |
| R²D-HOPE-MoRE (ours, medium) | ~11.5M | local checkpoint |
| GPT-2 small | 117M | HuggingFace |
| GPT-2 medium | 345M | HuggingFace |
| DistilGPT-2 | 82M | HuggingFace |
| OPT-125M | 125M | HuggingFace |
| Pythia-70M | 70M | HuggingFace |
| Pythia-160M | 160M | HuggingFace |

> **Why these baselines?** All are open-weights, run on T4 in fp16, and have published WikiText-103 perplexities to cross-check.
> Commercial APIs (GPT-4, Claude) are NOT included here because they cannot be benchmarked for throughput/VRAM.

## Cell 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/r2d_hope_training'

# Load r2d_hope package from Drive
import shutil
DRIVE_ZIP = '/content/drive/MyDrive/r2d_hope.zip'
if not os.path.exists('/content/r2d_hope'):
    if os.path.exists(DRIVE_ZIP):
        shutil.unpack_archive(DRIVE_ZIP, '/content')
    else:
        raise FileNotFoundError('Upload r2d_hope.zip to Drive root first. See COLAB_SETUP_FIX.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# Dependencies
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'datasets', 'tokenizers',
                'accelerate', 'matplotlib', 'pandas', 'tabulate'], check=True)
print('Setup complete.')

## Cell 1 — Benchmark Core (engine)

In [ ]:
import gc
import math
import time
import torch
import torch.nn as nn
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class BenchResult:
    name:            str
    params_m:        float          # millions
    ppl_wikitext:    float = float('inf')
    throughput_bs1:  float = 0.0    # tok/s at batch=1
    throughput_bs8:  float = 0.0    # tok/s at batch=8
    peak_vram_mb:    float = 0.0
    dtype:           str   = 'fp16'
    error:           str   = ''


def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


@torch.no_grad()
def measure_throughput(
    model: nn.Module,
    input_ids: torch.Tensor,
    forward_fn,
    n_warmup: int = 5,
    n_measure: int = 20,
) -> float:
    """Returns tokens/second."""
    B, S = input_ids.shape
    for _ in range(n_warmup):
        forward_fn(input_ids)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_measure):
        forward_fn(input_ids)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return (n_measure * B * S) / elapsed


def peak_vram_mb() -> float:
    return torch.cuda.max_memory_allocated() / 1e6


@torch.no_grad()
def compute_ppl_wikitext(
    forward_fn,           # callable(input_ids) -> logits [B, S, V]
    tokenizer,
    device: torch.device,
    dtype: torch.dtype,
    seq_len: int = 512,
    max_batches: int = 50,
    batch_size: int = 4,
) -> float:
    """Compute PPL on WikiText-103 validation split."""
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='validation', streaming=True)

    eos_id = tokenizer.eos_token_id or 0
    buffer: list[int] = []
    for item in ds:
        text = item.get('text', '').strip()
        if not text:
            continue
        ids = tokenizer.encode(text, add_special_tokens=False)
        buffer.extend(ids)
        buffer.append(eos_id)
        if len(buffer) >= seq_len * max_batches * batch_size:
            break

    total_nll = 0.0
    total_tok = 0
    n_batches = 0

    for i in range(0, len(buffer) - seq_len - 1, seq_len * batch_size):
        if n_batches >= max_batches:
            break
        chunks = []
        for b in range(batch_size):
            start = i + b * seq_len
            if start + seq_len + 1 > len(buffer):
                break
            chunks.append(buffer[start: start + seq_len + 1])
        if not chunks:
            break
        ids = torch.tensor(chunks, dtype=torch.long, device=device)
        input_ids = ids[:, :seq_len]
        labels    = ids[:, 1:seq_len + 1]

        with torch.autocast(device_type=device.type, dtype=dtype):
            logits = forward_fn(input_ids)  # [B, S, V]

        B, S, V = logits.shape
        loss = nn.functional.cross_entropy(
            logits.reshape(B * S, V),
            labels.reshape(B * S),
            reduction='sum',
        )
        total_nll += loss.item()
        total_tok += B * S
        n_batches += 1

    avg_nll = total_nll / max(1, total_tok)
    return math.exp(min(avg_nll, 20.0))


print('Benchmark engine loaded.')

## Cell 2 — Benchmark HuggingFace Baselines

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Benchmarking on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

HF_BASELINES = [
    ('DistilGPT-2',   'distilgpt2'),
    ('Pythia-70M',    'EleutherAI/pythia-70m'),
    ('GPT-2 Small',   'gpt2'),
    ('Pythia-160M',   'EleutherAI/pythia-160m'),
    ('OPT-125M',      'facebook/opt-125m'),
    ('GPT-2 Medium',  'gpt2-medium'),
]

SEQ_LEN   = 512
DTYPE     = torch.float16
results: list[BenchResult] = []

for name, model_id in HF_BASELINES:
    print(f'\n── {name} ({model_id}) ──')
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
        if tok.eos_token_id is None:
            tok.add_special_tokens({'eos_token': '</s>'})
        tok.pad_token = tok.eos_token

        mdl = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=DTYPE, device_map='auto'
        )
        mdl.eval()
        params = count_params_m(mdl)
        print(f'  Params: {params:.1f}M')

        # logits wrapper
        def hf_forward(ids, _mdl=mdl):
            return _mdl(ids).logits

        # PPL
        torch.cuda.reset_peak_memory_stats()
        ppl = compute_ppl_wikitext(
            hf_forward, tok, device, DTYPE,
            seq_len=SEQ_LEN, max_batches=30, batch_size=2
        )
        vram = peak_vram_mb()
        print(f'  PPL: {ppl:.2f}  VRAM: {vram:.0f} MB')

        # Throughput
        dummy_bs1 = torch.randint(0, 1000, (1, SEQ_LEN), device=device)
        dummy_bs8 = torch.randint(0, 1000, (8, SEQ_LEN), device=device)
        thr_bs1 = measure_throughput(mdl, dummy_bs1, hf_forward)
        thr_bs8 = measure_throughput(mdl, dummy_bs8, hf_forward)
        print(f'  Throughput: bs=1 → {thr_bs1:,.0f} tok/s | bs=8 → {thr_bs8:,.0f} tok/s')

        results.append(BenchResult(
            name=name, params_m=params,
            ppl_wikitext=ppl,
            throughput_bs1=thr_bs1, throughput_bs8=thr_bs8,
            peak_vram_mb=vram, dtype='fp16',
        ))

    except Exception as e:
        print(f'  ERROR: {e}')
        results.append(BenchResult(name=name, params_m=0, error=str(e)))

    finally:
        # Free GPU memory before next model
        try:
            del mdl
        except Exception:
            pass
        gc.collect()
        torch.cuda.empty_cache()

print('\nBaseline benchmarks complete.')

## Cell 3 — Benchmark R²D-HOPE-MoRE (Our Model)

In [ ]:
from r2d_hope import R2DConfig, R2D_HOPE_MoRE, build_or_load_tokenizer
from r2d_hope.trainer import evaluate

TOKENIZER_DIR = f'{DRIVE_ROOT}/tokenizer'

r2d_tokenizer = build_or_load_tokenizer(
    tokenizer_dir=TOKENIZER_DIR,
    vocab_size=16384,
)

OUR_MODELS = [
    ('R2D-HOPE-MoRE Small',
     R2DConfig(d_model=384, d_ffn=1024, d_embedding=192, num_heads=6, head_dim=64,
               vocab_size=16384, nested_depth=20, num_experts=4, top_k_experts=2,
               window_size=128, ddim_inference_steps=10),
     f'{DRIVE_ROOT}/checkpoints/r2d_hope_small'),
    ('R2D-HOPE-MoRE Medium',
     R2DConfig(d_model=512, d_ffn=1280, d_embedding=256, num_heads=8, head_dim=64,
               vocab_size=16384, nested_depth=20, num_experts=4, top_k_experts=2,
               window_size=128, ddim_inference_steps=10),
     f'{DRIVE_ROOT}/checkpoints/r2d_hope_medium'),
]

import glob

for name, cfg, ckpt_dir in OUR_MODELS:
    print(f'\n── {name} ──')
    try:
        mdl = R2D_HOPE_MoRE(cfg).to(device)

        # Load latest checkpoint if available
        ckpts = sorted(glob.glob(f'{ckpt_dir}/step_*.pt'))
        if ckpts:
            ckpt = torch.load(ckpts[-1], map_location=device)
            mdl.load_state_dict(ckpt['model'])
            print(f'  Loaded checkpoint: {ckpts[-1]}')
        else:
            print('  No checkpoint found — benchmarking random-init weights')

        mdl = mdl.to(DTYPE)
        mdl.eval()
        params = count_params_m(mdl)
        print(f'  Params: {params:.2f}M')

        # Build logits-only forward (R2D model forward needs ctx tensor)
        def r2d_forward(ids, _mdl=mdl, _cfg=cfg):
            B, S = ids.shape
            ctx = torch.zeros(B, 1, _cfg.d_model, device=device, dtype=DTYPE)
            # Use direct embedding + core to get logits without diffusion
            # For PPL we need token logits: run CE path
            out = _mdl(ids, ids, ctx)   # prompt=ids, answer=ids for logit extraction
            # Return ce_logits stored by model (shape [B, S, V])
            return _mdl._last_logits

        # PPL
        torch.cuda.reset_peak_memory_stats()
        ppl = compute_ppl_wikitext(
            r2d_forward, r2d_tokenizer, device, DTYPE,
            seq_len=128, max_batches=30, batch_size=4
        )
        vram = peak_vram_mb()
        print(f'  PPL: {ppl:.2f}  VRAM: {vram:.0f} MB')

        # Throughput (generate mode — DDIM)
        dummy_bs1 = torch.randint(1, cfg.vocab_size, (1, 8), device=device)
        dummy_bs8 = torch.randint(1, cfg.vocab_size, (8, 8), device=device)
        ctx_bs1   = torch.zeros(1, 1, cfg.d_model, device=device, dtype=DTYPE)
        ctx_bs8   = torch.zeros(8, 1, cfg.d_model, device=device, dtype=DTYPE)

        def gen_bs1(ids): return mdl.generate(ids, ctx_bs1, num_answer_tokens=32, ddim_steps=4)
        def gen_bs8(ids): return mdl.generate(ids, ctx_bs8, num_answer_tokens=32, ddim_steps=4)

        thr_bs1 = measure_throughput(mdl, dummy_bs1, gen_bs1, n_warmup=2, n_measure=10)
        thr_bs8 = measure_throughput(mdl, dummy_bs8, gen_bs8, n_warmup=2, n_measure=10)
        # Convert: each generate call produces num_answer_tokens per sample
        thr_bs1 = thr_bs1 * 32  # already per-call, multiply by tokens per call
        thr_bs8 = thr_bs8 * 32
        print(f'  Throughput (gen): bs=1 → {thr_bs1:,.0f} tok/s | bs=8 → {thr_bs8:,.0f} tok/s')

        results.append(BenchResult(
            name=name, params_m=params,
            ppl_wikitext=ppl,
            throughput_bs1=thr_bs1, throughput_bs8=thr_bs8,
            peak_vram_mb=vram, dtype='fp16',
        ))

    except Exception as e:
        import traceback; traceback.print_exc()
        results.append(BenchResult(name=name, params_m=0, error=str(e)))
    finally:
        try: del mdl
        except Exception: pass
        gc.collect()
        torch.cuda.empty_cache()

print('\nOur model benchmarks complete.')

## Cell 4 — Results Table

In [ ]:
import pandas as pd

rows = []
for r in results:
    if r.error:
        rows.append({
            'Model': r.name, 'Params (M)': '—',
            'PPL ↓': '—', 'PPL/param ↓': '—',
            'Throughput bs=1': '—', 'Throughput bs=8': '—',
            'VRAM (MB)': '—', 'Note': f'ERROR: {r.error[:40]}'
        })
    else:
        ppl_per_param = r.ppl_wikitext / r.params_m if r.params_m > 0 else float('inf')
        rows.append({
            'Model':              r.name,
            'Params (M)':         f'{r.params_m:.1f}',
            'PPL ↓':              f'{r.ppl_wikitext:.1f}',
            'PPL/param ↓':        f'{ppl_per_param:.2f}',
            'Throughput bs=1':    f'{r.throughput_bs1:,.0f}',
            'Throughput bs=8':    f'{r.throughput_bs8:,.0f}',
            'VRAM (MB)':          f'{r.peak_vram_mb:.0f}',
            'Note':               ''
        })

df = pd.DataFrame(rows)
# Sort by PPL ascending (best first), errors last
df['_sort'] = pd.to_numeric(df['PPL ↓'], errors='coerce')
df = df.sort_values('_sort').drop(columns=['_sort'])

print('\n' + '='*90)
print('  BENCHMARK RESULTS — R²D-HOPE-MoRE vs SOTA Open-Weight Models')
print('='*90)
print(df.to_string(index=False))
print('='*90)
print('\nPPL/param: lower is better (measures parameter efficiency)')
print('PPL:       lower is better (measures language modelling quality)')
print('All PPL measured on WikiText-103 validation, seq_len=512, fp16')

## Cell 5 — Visualise Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

valid = [r for r in results if not r.error and r.ppl_wikitext < 1e6]

names  = [r.name for r in valid]
ppls   = [r.ppl_wikitext for r in valid]
params = [r.params_m for r in valid]
tputs  = [r.throughput_bs8 for r in valid]
colors = ['#e63946' if 'R2D' in n else '#457b9d' for n in names]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('R²D-HOPE-MoRE vs SOTA Open-Weight Models', fontsize=14, fontweight='bold')

# Plot 1: PPL bar chart
ax = axes[0]
bars = ax.barh(names, ppls, color=colors)
ax.set_xlabel('Perplexity (↓ better)')
ax.set_title('WikiText-103 Perplexity')
ax.invert_yaxis()
for bar, val in zip(bars, ppls):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=8)

# Plot 2: PPL vs Params scatter
ax = axes[1]
for n, p, ppl, c in zip(names, params, ppls, colors):
    ax.scatter(p, ppl, color=c, s=120, zorder=3)
    ax.annotate(n, (p, ppl), textcoords='offset points', xytext=(5, 3), fontsize=7)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Perplexity (↓ better)')
ax.set_title('PPL vs Parameter Count\n(lower-left = best)')
ax.grid(True, alpha=0.3)

# Plot 3: Throughput bar
ax = axes[2]
bars = ax.barh(names, tputs, color=colors)
ax.set_xlabel('Tokens/sec at batch=8 (↑ better)')
ax.set_title('Generation Throughput (bs=8)')
ax.invert_yaxis()
for bar, val in zip(bars, tputs):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=8)

ours_patch  = mpatches.Patch(color='#e63946', label='R²D-HOPE-MoRE (ours)')
base_patch  = mpatches.Patch(color='#457b9d', label='Baselines')
fig.legend(handles=[ours_patch, base_patch], loc='lower center', ncol=2, fontsize=9)

plt.tight_layout(rect=[0, 0.05, 1, 1])
out_path = f'{DRIVE_ROOT}/benchmark_results.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {out_path}')

## Cell 6 — Save Results CSV

In [ ]:
import json, os

# Save as CSV
csv_path = f'{DRIVE_ROOT}/benchmark_results.csv'
df.to_csv(csv_path, index=False)
print(f'CSV saved to {csv_path}')

# Save raw results as JSON for programmatic comparison later
json_path = f'{DRIVE_ROOT}/benchmark_results.json'
with open(json_path, 'w') as f:
    json.dump([
        {
            'name': r.name, 'params_m': r.params_m,
            'ppl_wikitext': r.ppl_wikitext,
            'throughput_bs1': r.throughput_bs1,
            'throughput_bs8': r.throughput_bs8,
            'peak_vram_mb': r.peak_vram_mb,
            'dtype': r.dtype, 'error': r.error,
        } for r in results
    ], f, indent=2)
print(f'JSON saved to {json_path}')

## Notes on Fair Comparison

- **PPL is not fully apples-to-apples** between R²D-HOPE-MoRE and standard AR models because R²D uses a diffusion objective (noise prediction), not direct next-token prediction. The PPL reported for our model is via the CE head, which is the closest comparable metric.
- **Throughput for R²D includes DDIM steps** (`ddim_steps=4` here). Standard AR models generate 1 token per forward pass. R²D generates `num_answer_tokens` in parallel across DDIM steps — direct latency comparison requires normalising by output length.
- **Parameter efficiency** (PPL/param) is the most honest metric given the 6.5M vs 117M+ size gap.
- Expected result: our model will have higher raw PPL than GPT-2 (fewer params), but dramatically better PPL/param and throughput at small batch sizes.